# LLM SVG Generation Benchmark

Compare how multiple LLMs translate a text prompt into SVG markup using OpenRouter.

In [ ]:
import os
import re
import time
import xml.etree.ElementTree as ET
from datetime import datetime

from dotenv import load_dotenv
from IPython.display import SVG, display
from openai import OpenAI

load_dotenv()

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
API_KEY = os.getenv("OPENROUTER_API_KEY")

if not API_KEY:
    raise EnvironmentError("Set OPENROUTER_API_KEY in your environment or .env file.")

client = OpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=API_KEY,
)

In [ ]:
challenge = "a panda rollerblading to work"
prompt = (
    f"Generate an SVG illustration of {challenge}. "
    "Return valid SVG markup only, without Markdown fences or explanations."
)

messages = [{"role": "user", "content": prompt}]

In [ ]:
models = [
    ("openai/gpt-oss-120b", None),
    ("openai/gpt-6-luna", "low"),
    ("deepseek/deepseek-v3.2", None),
    ("moonshotai/kimi-k2-thinking", None),
    ("x-ai/grok-4.3", None),
    ("anthropic/claude-opus-4.5", None),
    ("openai/gpt-5.2", "high"),
    ("google/gemini-4-argon", None),
]

In [ ]:
def generate_svg(model, reasoning_effort=None):
    """Request SVG markup from a model and return its output and latency."""
    request = {
        "model": model,
        "messages": messages,
    }
    if reasoning_effort:
        request["reasoning_effort"] = reasoning_effort

    started = time.perf_counter()
    response = client.chat.completions.create(**request)
    elapsed = time.perf_counter() - started

    content = response.choices[0].message.content or ""
    return content.strip(), elapsed

In [ ]:
def sanitize_svg(svg_text):
    """Allow a small set of static SVG elements and attributes."""
    svg_text = re.sub(r"^```(?:svg|xml)?\s*|\s*```$", "", svg_text.strip(), flags=re.IGNORECASE)
    start = svg_text.find("<svg")
    if start == -1:
        raise ValueError("No SVG root element found in model output.")

    svg_text = svg_text[start:]
    root = ET.fromstring(svg_text)

    allowed_tags = {
        "svg", "g", "path", "rect", "circle", "ellipse", "line",
        "polyline", "polygon", "text", "tspan", "defs", "linearGradient",
        "radialGradient", "stop", "clipPath", "mask", "title", "desc"
    }
    allowed_attrs = {
        "xmlns", "viewBox", "width", "height", "x", "y", "x1", "y1",
        "x2", "y2", "cx", "cy", "r", "rx", "ry", "d", "points",
        "fill", "fill-opacity", "stroke", "stroke-width", "stroke-linecap",
        "stroke-linejoin", "stroke-opacity", "opacity", "transform",
        "font-size", "font-family", "font-weight", "text-anchor",
        "offset", "stop-color", "stop-opacity", "id", "clip-path", "mask"
    }

    def clean(element):
        tag = element.tag.split("}")[-1]
        if tag not in allowed_tags:
            element.clear()
            return False

        for attribute in list(element.attrib):
            name = attribute.split("}")[-1]
            value = element.attrib[attribute].strip()
            if name not in allowed_attrs or re.search(r"url\s*\(|javascript:|https?://", value, re.I):
                del element.attrib[attribute]

        for child in list(element):
            if not clean(child):
                element.remove(child)
        return True

    if not clean(root):
        raise ValueError("Unsupported SVG root element.")

    return ET.tostring(root, encoding="unicode")

In [ ]:
results = []

for model, effort in models:
    try:
        svg_markup, elapsed = generate_svg(model, effort)
        results.append({
            "model": model,
            "svg": svg_markup,
            "seconds": elapsed,
            "error": None,
        })
        print(f"{model}: {elapsed:.1f}s")
    except Exception as exc:
        results.append({
            "model": model,
            "svg": None,
            "seconds": None,
            "error": str(exc),
        })
        print(f"{model}: failed — {exc}")

In [ ]:
for result in results:
    print(f"{result['model']}")

    if result["error"]:
        print(f"Error: {result['error']}")
        continue

    try:
        display(SVG(sanitize_svg(result["svg"])))
    except Exception as exc:
        print(f"Could not render SVG: {exc}")

## Notes

- Model availability and identifiers depend on OpenRouter.
- API usage may incur charges.
- SVG output is filtered before rendering to remove unsupported elements and attributes.